# Updated Emotion Training Notebook

This notebook uses the same pipeline as `train_emotion_models.py`, but in a notebook-friendly flow.

In [ ]:
from argparse import Namespace
from pathlib import Path
import train_emotion_models as tem

In [ ]:
args = Namespace(
    data_root=None,
    output_dir='outputs',
    models=['cnn', 'mobilenet', 'efficientnet'],
    img_size=48,
    img_size_tl=128,
    batch_size=0,
    val_split=0.15,
    epochs_cnn=40,
    epochs_head=10,
    epochs_finetune=18,
    label_smoothing=0.1,
)

args

## Run Training

Edit the `args` cell above if you want fewer models or fewer epochs before running this cell.

In [ ]:
tem.set_reproducibility()
gpu_available = tem.configure_runtime()

batch_size = args.batch_size or (64 if gpu_available else 32)
output_dir = Path(args.output_dir)
output_dir.mkdir(parents=True, exist_ok=True)

train_dir, test_dir = tem.resolve_data_root(args.data_root)
tem.save_dataset_report(train_dir, test_dir, output_dir)
tem.save_sample_grid(train_dir, output_dir)

gray_bundle = tem.build_dataset_bundle(train_dir, test_dir, args.img_size, batch_size, 'grayscale', args.val_split)
rgb_bundle = tem.build_dataset_bundle(train_dir, test_dir, args.img_size_tl, batch_size, 'rgb', args.val_split)

results = []
histories = {}

if 'cnn' in args.models:
    model, history = tem.train_custom_cnn(gray_bundle, args, output_dir)
    results.append(tem.evaluate_model(model, gray_bundle['test'], 'Custom CNN', output_dir))
    histories['custom_cnn'] = history

if 'mobilenet' in args.models:
    model, history = tem.train_mobilenet(rgb_bundle, args, output_dir)
    results.append(tem.evaluate_model(model, rgb_bundle['test'], 'MobileNetV2', output_dir))
    histories['mobilenet'] = history

if 'efficientnet' in args.models:
    model, history = tem.train_efficientnet(rgb_bundle, args, output_dir)
    results.append(tem.evaluate_model(model, rgb_bundle['test'], 'EfficientNetB0', output_dir))
    histories['efficientnet'] = history

tem.save_summary(results, histories, output_dir)

best_result = max(results, key=lambda item: item['test_acc'])
print('\nFinal Results')
print('=' * 60)
for result in sorted(results, key=lambda item: item['test_acc'], reverse=True):
    print(f"{result['model_name']:15s} acc={result['test_acc'] * 100:.2f}%  f1={result['macro_f1']:.4f}")
print('=' * 60)
print(f"Best model: {best_result['model_name']} ({best_result['test_acc'] * 100:.2f}%)")
print(f"Outputs saved to: {output_dir.resolve()}")